# Phase 6 — VPD-lite Training
**Contract-Guided Variational Policy Distillation** on Google Colab

### Setup checklist
Upload to your Google Drive under `MyDrive/telco_vpd/`:
```
telco_vpd/
  adapters/
    qwen3-4b-feedback-sdft/
      adapters.safetensors      ← from outputs/sft_mlx/qwen3-4b-feedback-sdft/
      adapter_config.json
  data/
    sdpo_rollouts.jsonl           ← from data/sdpo_rollouts.jsonl
```

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q transformers>=4.51.0 peft>=0.15.0 accelerate safetensors bitsandbytes

In [ ]:
# ── 2. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/telco_vpd'
MLX_ADAPTER_DIR = f'{DRIVE_BASE}/adapters/qwen3-4b-feedback-sdft'
ROLLOUTS_FILE   = f'{DRIVE_BASE}/data/sdpo_rollouts.jsonl'
OUTPUT_DIR      = f'{DRIVE_BASE}/adapters/qwen3-4b-vpd'
MODEL_ID        = 'Qwen/Qwen3-4B'

In [ ]:
# ── 3. Convert MLX adapter → HuggingFace PEFT ────────────────────────────────
# MLX LoRA key format:  model.layers.{i}.{module}.lora_a  shape (in, rank)
# PEFT key format:      base_model.model.model.layers.{i}.{module}.lora_A.weight  shape (rank, in)
# Both lora_a and lora_b need to be transposed.

import json
from pathlib import Path
import safetensors.torch as st
import torch

PEFT_ADAPTER_DIR = '/content/peft_m3'

def convert_mlx_to_peft(mlx_dir: str, out_dir: str) -> None:
    mlx_dir, out_dir = Path(mlx_dir), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Load MLX safetensors
    raw = {}
    with st.safe_open(str(mlx_dir / 'adapters.safetensors'), framework='pt') as f:
        for k in f.keys():
            raw[k] = f.get_tensor(k)

    # Convert: rename + transpose
    peft_weights = {}
    for k, t in raw.items():
        # model.layers.X.mod.lora_a → base_model.model.model.layers.X.mod.lora_A.weight
        new_k = 'base_model.model.' + k
        if new_k.endswith('.lora_a'):
            new_k = new_k[:-7] + '.lora_A.weight'
            t = t.T.contiguous()   # (in, rank) → (rank, in)
        elif new_k.endswith('.lora_b'):
            new_k = new_k[:-7] + '.lora_B.weight'
            t = t.T.contiguous()   # (rank, out) → (out, rank)
        peft_weights[new_k] = t

    st.save_file(peft_weights, str(out_dir / 'adapter_model.safetensors'))

    # PEFT adapter_config.json
    # lora_alpha=20 → alpha/r = 20/8 = 2.5 = mlx scale(20)/r(8)
    # init_lora_weights=False: don't zero-init lora_B — weights come from the file
    peft_cfg = {
        'peft_type': 'LORA',
        'task_type': 'CAUSAL_LM',
        'base_model_name_or_path': MODEL_ID,
        'r': 8,
        'lora_alpha': 20,
        'lora_dropout': 0.0,
        'bias': 'none',
        'target_modules': ['q_proj','k_proj','v_proj','o_proj',
                           'gate_proj','up_proj','down_proj'],
        'layers_to_transform': list(range(28, 36)),  # last 8 of 36 layers
        'inference_mode': False,
        'init_lora_weights': False,   # must be False: we load weights from file, not random-init
    }
    with open(out_dir / 'adapter_config.json', 'w') as f:
        json.dump(peft_cfg, f, indent=2)

    print(f'Converted {len(peft_weights)} tensors → {out_dir}')
    print('Sample keys:', list(peft_weights.keys())[:2])

convert_mlx_to_peft(MLX_ADAPTER_DIR, PEFT_ADAPTER_DIR)

In [ ]:
# ── 4. Load base model + teacher/student adapters ─────────────────────────────
# T4 (15 GB): QLoRA — base in 4-bit (frozen), LoRA weights in fp32 (trainable)
# A100 (40 GB): remove bnb_cfg + prepare_model_for_kbit_training, use bfloat16

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = 'right'

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

# Required for QLoRA: enables gradient checkpointing + marks base params non-trainable
# so only LoRA weights receive gradients
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)

# Load M3 as 'teacher'
model = PeftModel.from_pretrained(base, PEFT_ADAPTER_DIR,
                                   adapter_name='teacher', is_trainable=True)
# Load same M3 weights as 'student', also trainable
model.load_adapter(PEFT_ADAPTER_DIR, adapter_name='student', is_trainable=True)

model.print_trainable_parameters()

In [ ]:
# ── 5. Load rollout data ──────────────────────────────────────────────────────
import json

TEACHER_SUFFIX = (
    '\n\n[SDPO: Previous Attempt]\n{best_response}'
    '\n\n[SDPO: Environment Feedback]\n{feedback_text}'
)

def fmt_feedback(fb):
    texts = fb.get('feedback_text', [])
    if texts: return ' '.join(texts)
    errors = fb.get('errors', [])
    if errors: return '; '.join(e.get('type', str(e)) for e in errors)
    return 'The response was incorrect.'

def build_teacher_msgs(record, rollout_idx):
    """Teacher context for a specific rollout: shows THAT rollout's response as demo."""
    ro = record['rollouts'][rollout_idx]
    raw = ro.get('raw_output', '') or json.dumps(ro['prediction'])
    fb  = fmt_feedback(ro['feedback'])
    msgs = record['prompt']
    sys_content = msgs[0]['content'] + TEACHER_SUFFIX.format(
        best_response=raw, feedback_text=fb)
    return [{'role': 'system', 'content': sys_content}] + msgs[1:]

rollout_records = [json.loads(l) for l in open(ROLLOUTS_FILE) if l.strip()]

# VPD training data: ONE sample per prompt = best (positive) rollout only.
# Teacher context shows THAT best response as demo — consistent with what M-step trains on.
# Negative rollouts are kept separately for E-step's contrastive penalty.
samples  = []   # positive — one per prompt group, used for both E-step and M-step
neg_bank = []   # negative rollouts from all prompts, sampled randomly in E-step

for rec in rollout_records:
    best_idx = rec['best_rollout_idx']
    student_msgs = rec['prompt']   # original prompt, no feedback

    # Positive sample: best rollout (skip prompt groups where all K rollouts failed)
    if best_idx >= 0:
        best_ro  = rec['rollouts'][best_idx]
        raw      = best_ro.get('raw_output', '') or json.dumps(best_ro['prediction'])
        if raw.strip():
            samples.append({
                'id':           rec['id'],
                'student_msgs': student_msgs,
                'teacher_msgs': build_teacher_msgs(rec, best_idx),  # demo = THIS response
                'response_raw': raw,
                'reward':       best_ro['reward'],
            })

    # Negative bank: failed rollouts for E-step contrastive penalty.
    # student_msgs stored so E-step can compute π_θ(y-|x) reference correctly.
    for k, ro in enumerate(rec['rollouts']):
        if ro['reward'] < 1.0:
            raw = ro.get('raw_output', '') or json.dumps(ro['prediction'])
            if raw.strip():
                neg_bank.append({
                    'student_msgs': student_msgs,                   # prompt only (no feedback)
                    'teacher_msgs': build_teacher_msgs(rec, k),     # demo = THIS failed response
                    'response_raw': raw,
                })

print(f'Positive training samples: {len(samples)} (one per prompt)')
print(f'Negative bank for E-step:  {len(neg_bank)}')

In [ ]:
# ── 6. Tokenisation helpers ───────────────────────────────────────────────────
import torch

DEVICE = next(model.parameters()).device
EOS = tokenizer.eos_token or ''
MAX_SEQ_LEN = 1024   # truncate long tool-schema contexts to fit in GPU memory

def tokenize_sample(sample, use_teacher_ctx=True):
    msgs = sample['teacher_msgs'] if use_teacher_ctx else sample['student_msgs']
    ctx  = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    resp = sample['response_raw'].strip() + EOS

    ctx_ids  = tokenizer.encode(ctx,  add_special_tokens=False)
    resp_ids = tokenizer.encode(resp, add_special_tokens=False)

    # Truncate context from the LEFT (keep the most recent tokens = user turn)
    max_ctx = MAX_SEQ_LEN - len(resp_ids) - 10
    if len(ctx_ids) > max_ctx:
        ctx_ids = ctx_ids[-max_ctx:]

    return ctx_ids, resp_ids

In [ ]:
# ── 7. Loss functions ─────────────────────────────────────────────────────────
import torch.nn.functional as F
import math

def compute_seq_logprob(logits, ids_list):
    """Sum of log-probs across response tokens. logits: (T, V), ids_list: list[int]."""
    ids = torch.tensor(ids_list, dtype=torch.long, device=logits.device)  # (T,)
    lp  = F.log_softmax(logits.float(), dim=-1)                            # (T, V)
    return lp.gather(1, ids.unsqueeze(1)).squeeze(1).sum()                 # scalar


def compute_jsd_loss(student_logits, teacher_logits, top_k=20, alpha=0.5):
    """Top-k JSD distillation. Expects (n_resp, V) float32 logits."""
    t_lp = F.log_softmax(teacher_logits.float(), dim=-1)
    s_lp = F.log_softmax(student_logits.float(),  dim=-1)

    t_topk_lp, t_topk_idx = t_lp.topk(top_k, dim=-1)
    s_topk_lp = s_lp.gather(-1, t_topk_idx)

    s_topk_lp = s_topk_lp - torch.logsumexp(s_topk_lp, -1, keepdim=True)
    t_topk_lp = t_topk_lp - torch.logsumexp(t_topk_lp, -1, keepdim=True)

    log_M = torch.logaddexp(
        s_topk_lp + math.log(1 - alpha + 1e-8),
        t_topk_lp + math.log(alpha      + 1e-8),
    )
    kl_s = (s_topk_lp.exp() * (s_topk_lp - log_M)).sum(-1)
    kl_t = (t_topk_lp.exp() * (t_topk_lp - log_M)).sum(-1)
    return ((1 - alpha) * kl_s + alpha * kl_t).mean()


def kl_approx(teacher_logits, student_logits_frozen, top_k=50):
    """KL(teacher || student) on teacher top-k tokens."""
    t_lp = F.log_softmax(teacher_logits.float(), dim=-1)
    s_lp = F.log_softmax(student_logits_frozen.float(), dim=-1)
    t_topk_lp, t_topk_idx = t_lp.topk(top_k, dim=-1)
    s_topk_lp = s_lp.gather(-1, t_topk_idx)
    return (t_topk_lp.exp() * (t_topk_lp - s_topk_lp)).sum(-1).mean()

In [ ]:
# ── 8. VPD-lite training loop ─────────────────────────────────────────────────
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

CFG = dict(
    lr_e    = 5e-4,  # 50× larger than before — teacher must diverge within 3 epochs
    lr_m    = 5e-6,
    epochs  = 3,
    m_per_e = 4,     # 1 E-step per 4 M-steps
    beta    = 0.1,   # DPO literature uses 0.1; β=0.02 = flat loss landscape (teacher barely moves)
    top_k   = 20,
    alpha   = 0.5,
)

# ── Grab per-adapter params directly from PEFT's ModuleDict ───────────────────
teacher_params, student_params = [], []
for _, module in model.named_modules():
    if hasattr(module, 'lora_A') and isinstance(module.lora_A, nn.ModuleDict):
        for name, linear in module.lora_A.items():
            p = list(linear.parameters()) + list(module.lora_B[name].parameters())
            if name == 'teacher': teacher_params.extend(p)
            elif name == 'student': student_params.extend(p)

for p in teacher_params + student_params:
    p.requires_grad_(True)

assert teacher_params[0] is not student_params[0], 'ERROR: shared tensors!'
print(f'Teacher: {sum(p.numel() for p in teacher_params):,}  '
      f'Student: {sum(p.numel() for p in student_params):,}')

opt_e = AdamW(teacher_params, lr=CFG['lr_e'])
opt_m = AdamW(student_params, lr=CFG['lr_m'])

# ── Adapter-routing sanity check ──────────────────────────────────────────────
# Both adapters start identical.  After the first E-step, teacher must differ.
# If max_logit_diff stays 0 after E-step, set_adapter() routing is broken.
_probe = torch.zeros(1, 5, dtype=torch.long, device=DEVICE)
with torch.no_grad():
    model.set_adapter('teacher'); _t0 = model(_probe).logits[0, -1, :5].clone()
    model.set_adapter('student'); _s0 = model(_probe).logits[0, -1, :5].clone()
print(f'[init] teacher == student: {torch.allclose(_t0, _s0)}  (expected True)')

# ── Helpers ───────────────────────────────────────────────────────────────────
def forward(model, ctx_ids, resp_ids):
    """Forward pass → response logits (n_resp, V)."""
    ids = torch.tensor(ctx_ids + resp_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    out = model(ids)
    logits = out.logits[0]
    resp_start = len(ctx_ids) - 1
    result = logits[resp_start : resp_start + len(resp_ids)].clone()
    del ids, out, logits
    return result

def e_step(pos_sample):
    """
    VPD E-step: DPO-style binary classifier on implicit rewards (VPD paper Eq. 5).

    Implicit reward:  r̃ = β · [log q_φ(y|x,C) − log π_θ(y|x)]
                              teacher-with-feedback   student-prompt-only (no grad)
    Loss = −log σ(r̃_pos − δ) − log σ(−(r̃_neg − δ))
    where δ = ½(r̃_pos + r̃_neg)  (centering margin, no grad).
    """
    # ── Positive trajectory ────────────────────────────────────────────────
    model.set_adapter('teacher')
    t_ctx_pos, resp_pos = tokenize_sample(pos_sample, use_teacher_ctx=True)
    t_logits_pos        = forward(model, t_ctx_pos, resp_pos)
    t_lp_pos            = compute_seq_logprob(t_logits_pos, resp_pos)   # grad ← teacher

    with torch.no_grad():
        model.set_adapter('student')
        s_ctx_pos, _ = tokenize_sample(pos_sample, use_teacher_ctx=False)  # prompt only
        s_logits_pos = forward(model, s_ctx_pos, resp_pos)
        s_lp_pos     = compute_seq_logprob(s_logits_pos, resp_pos)
    model.set_adapter('teacher')   # restore BEFORE backward

    r_tilde_pos = CFG['beta'] * (t_lp_pos - s_lp_pos)

    # ── Negative trajectory ────────────────────────────────────────────────
    if neg_bank:
        neg_s = random.choice(neg_bank)
        t_ctx_neg, resp_neg = tokenize_sample(neg_s, use_teacher_ctx=True)
        t_logits_neg        = forward(model, t_ctx_neg, resp_neg)
        t_lp_neg            = compute_seq_logprob(t_logits_neg, resp_neg)

        with torch.no_grad():
            model.set_adapter('student')
            s_ctx_neg, _ = tokenize_sample(neg_s, use_teacher_ctx=False)
            s_logits_neg = forward(model, s_ctx_neg, resp_neg)
            s_lp_neg     = compute_seq_logprob(s_logits_neg, resp_neg)
        model.set_adapter('teacher')

        r_tilde_neg = CFG['beta'] * (t_lp_neg - s_lp_neg)
        delta       = 0.5 * (r_tilde_pos.detach() + r_tilde_neg.detach())
        loss        = -F.logsigmoid( r_tilde_pos - delta) \
                      -F.logsigmoid(-(r_tilde_neg - delta))
        del t_logits_neg, s_logits_neg
    else:
        loss = -t_lp_pos / max(len(resp_pos), 1)   # no negatives: CE only

    opt_e.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(teacher_params, 1.0)
    opt_e.step()
    l = loss.item()
    del t_logits_pos, s_logits_pos, loss
    torch.cuda.empty_cache()
    return l

def m_step(pos_sample):
    """
    VPD M-step: distil updated teacher into student via top-k JSD.
    Teacher sees: prompt + feedback + demo response (full context C).
    Student sees: prompt only.  Both score the SAME response tokens.
    """
    s_ctx_ids, resp_ids = tokenize_sample(pos_sample, use_teacher_ctx=False)
    t_ctx_ids, _        = tokenize_sample(pos_sample, use_teacher_ctx=True)

    with torch.no_grad():
        model.set_adapter('teacher')
        t_logits = forward(model, t_ctx_ids, resp_ids)

    model.set_adapter('student')
    s_logits = forward(model, s_ctx_ids, resp_ids)

    loss = compute_jsd_loss(s_logits, t_logits, CFG['top_k'], CFG['alpha'])
    opt_m.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(student_params, 1.0)
    opt_m.step()
    l = loss.item()
    del t_logits, s_logits, loss
    torch.cuda.empty_cache()
    return l

# ── Training loop ─────────────────────────────────────────────────────────────
best_m_loss  = float('inf')
_e_step_done = 0

for epoch in range(CFG['epochs']):
    random.shuffle(samples)
    e_losses, m_losses = [], []

    for i, sample in enumerate(samples):
        if i % (CFG['m_per_e'] + 1) == 0:
            e_losses.append(e_step(sample))
            _e_step_done += 1

            # After 1st E-step: verify teacher actually diverged from student
            if _e_step_done == 1:
                with torch.no_grad():
                    model.set_adapter('teacher'); _t1 = model(_probe).logits[0, -1, :5].clone()
                    model.set_adapter('student'); _s1 = model(_probe).logits[0, -1, :5].clone()
                diff = (_t1 - _s1).abs().max().item()
                print(f'[after 1st E-step] teacher≠student: {not torch.allclose(_t1,_s1,atol=1e-5)}'
                      f'  max_logit_diff={diff:.8f}')
        else:
            m_losses.append(m_step(sample))

        if (i + 1) % 10 == 0:
            avg_e = sum(e_losses[-5:])  / max(len(e_losses[-5:]), 1)
            avg_m = sum(m_losses[-10:]) / max(len(m_losses[-10:]), 1)
            print(f'  epoch={epoch+1}  step={i+1}/{len(samples)}  '
                  f'E={avg_e:.4f}  M={avg_m:.6f}')   # 6 dp to see small non-zero values

    avg_m = sum(m_losses) / max(len(m_losses), 1)
    avg_e = sum(e_losses) / max(len(e_losses), 1)
    print(f'Epoch {epoch+1}/{CFG["epochs"]}  E_avg={avg_e:.4f}  M_avg={avg_m:.6f}')
    if avg_m < best_m_loss:
        best_m_loss = avg_m
        print('  → best M-loss so far')

print('Training done.')

In [ ]:
# ── 9. Save student adapter → Google Drive ────────────────────────────────────
import shutil
from pathlib import Path
import safetensors.torch as st

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# In a multi-adapter PEFT model, named_parameters() includes the adapter name:
#   base_model.model.model.layers.28.mlp.down_proj.lora_A.student.weight
# Filter by adapter name in the key path — do NOT rely on p.requires_grad,
# which may be False for student if the last training step was an E-step.
student_weights = {
    n: p.data for n, p in model.named_parameters()
    if '.lora_A.student.' in n or '.lora_B.student.' in n
}
print(f'Student params found: {len(student_weights)}')
assert len(student_weights) > 0, 'ERROR: no student params — check adapter name in named_parameters()'

# Convert PEFT multi-adapter keys back to MLX format:
#   base_model.model.model.layers.X.mod.lora_A.student.weight → model.layers.X.mod.lora_a
mlx_weights = {}
for k, t in student_weights.items():
    k2 = k.replace('base_model.model.', '')
    if '.lora_A.student.weight' in k2:
        k2 = k2.replace('.lora_A.student.weight', '.lora_a')
        t = t.T.contiguous()   # (rank, in) → (in, rank)
    elif '.lora_B.student.weight' in k2:
        k2 = k2.replace('.lora_B.student.weight', '.lora_b')
        t = t.T.contiguous()   # (out, rank) → (rank, out)
    mlx_weights[k2] = t.to(torch.float32)

print(f'MLX tensors to save: {len(mlx_weights)}')
assert len(mlx_weights) == len(student_weights), 'Key conversion missed some tensors!'

st.save_file(mlx_weights, str(out_dir / 'adapters.safetensors'))

# Copy MLX-format adapter_config.json from M3 (for local eval with mlx-lm)
src_cfg = Path(MLX_ADAPTER_DIR) / 'adapter_config.json'
if src_cfg.exists():
    shutil.copy2(src_cfg, out_dir / 'adapter_config.json')

print(f'Saved → {out_dir}')
print('Sample MLX keys:', list(mlx_weights.keys())[:3])
print()
print('Download adapter and eval locally:')
print('  python3.11 scripts/run_baseline.py --backend mlx --model Qwen/Qwen3-4B \\')
print('      --adapter outputs/vpd/qwen3-4b-vpd')

## After training

1. Download `adapters.safetensors` + `adapter_config.json` from Drive
2. Place into `outputs/vpd/qwen3-4b-vpd/`
3. Run local eval:
```bash
python3.11 scripts/run_baseline.py \\
    --backend mlx --model Qwen/Qwen3-4B \\
    --adapter outputs/vpd/qwen3-4b-vpd
```